# Package and submit the current agent

This notebook validates the required submission files, builds the official tarball layout, and submits it to Kaggle through the authenticated API. It never embeds or prints credentials.

Submission is intentionally disabled until `CONFIRM_SUBMIT` is changed to `True`.

In [ ]:
from pathlib import Path
import json, os, tarfile

COMPETITION = 'pokemon-tcg-ai-battle'
MESSAGE = 'Mega Lucario ex agent'
CONFIRM_SUBMIT = False  # Change to True only when ready to use one submission attempt.

ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
SUBMISSION = ROOT / 'submission'
TARBALL = ROOT / 'submission.tar.gz'
required = [SUBMISSION / 'main.py', SUBMISSION / 'deck.csv', SUBMISSION / 'cg']
missing = [str(p.relative_to(ROOT)) for p in required if not p.exists()]
if missing: raise FileNotFoundError('Missing required submission files: ' + ', '.join(missing))
print('Repository:', ROOT)
print('Submission files are present')

In [ ]:
# Build exactly main.py, deck.csv, and cg/ at the tarball root.
with tarfile.open(TARBALL, 'w:gz') as archive:
    archive.add(SUBMISSION / 'main.py', arcname='main.py')
    archive.add(SUBMISSION / 'deck.csv', arcname='deck.csv')
    archive.add(SUBMISSION / 'cg', arcname='cg', recursive=True)

with tarfile.open(TARBALL, 'r:gz') as archive:
    members = archive.getnames()
print('Created:', TARBALL)
print('Tarball size:', round(TARBALL.stat().st_size / 1_000_000, 2), 'MB')
print('Top-level entries:', sorted({m.split('/')[0] for m in members}))
assert 'main.py' in members and 'deck.csv' in members and any(m.startswith('cg/') for m in members)
assert all(not m.startswith(('.env', 'kaggle.json')) for m in members)

## Submit

The next cell is the only cell that submits to Kaggle. Leave the confirmation flag disabled for packaging-only checks.

In [ ]:
if not CONFIRM_SUBMIT:
    print('Packaging complete. Submission not sent; set CONFIRM_SUBMIT = True to continue.')
else:
    token_file = Path.home() / '.kaggle' / 'access_token'
    if token_file.exists():
        os.environ.setdefault('KAGGLE_API_TOKEN', token_file.read_text().strip())
    from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi()
    api.authenticate()
    response = api.competition_submit(
        file_name=str(TARBALL), message=MESSAGE, competition=COMPETITION
    )
    print('Submission request sent.')
    print('Reference:', getattr(response, 'ref', None))
    print('Status:', getattr(response, 'status', None))

The existing equivalent command-line workflow is: `./tools/submit.sh "your message"`. Kaggle submission limits still apply, so run local evaluation before enabling submission.